In [1]:

%pip install \
    "strands-agents>=1.60.0,<2.0.0" \
    "strands-agents-tools>=0.8.9,<0.9.0" \
    "langchain>=1.4.0,<2.0.0" \
    "langchain-aws>=1.8.0,<2.0.0" \
    "langgraph>=1.2.0,<2.0.0" \
    "boto3>=1.40.0"


INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.5/923.5 kB 8.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 43.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 85.0 MB/s  0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-con

In [1]:

from strands import Agent
from strands.models.openai import OpenAIModel
import os
from google.colab import userdata

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": userdata.get("GROQ_API_KEY"),
        "base_url": os.getenv(
            "GROQ_BASE_URL",
            "https://api.groq.com/openai/v1"
        ),
        "max_retries": 5,
    },
    model_id=MODEL_ID,
    params={"extra_body": {"include_reasoning": False}},
)

agent = Agent(model=model, callback_handler=None )

response = agent(
    """
      Explain groq in a bulleted list.
    """
)

print(response)


**GROQ – “Graph‑Relational Object Queries” (Sanity.io’s query language)**  

- **What it is**  
  - A powerful, JSON‑centric query language created by Sanity for fetching and shaping content stored in a Sanity dataset.  
  - Designed to be expressive like GraphQL but with a simpler, more flexible syntax that works directly on the raw document structure.

- **Core concepts**  
  - **Documents** – top‑level objects in Sanity (e.g., `post`, `author`, `product`).  
  - **Fields** – properties inside a document; can be primitives, arrays, objects, or references to other documents.  
  - **References** – pointers (`_ref`) that link one document to another, enabling relational queries.  
  - **Projection** – the part of the query that selects which fields to return (similar to a “select” clause).  

- **Basic syntax**  
  - `*` – selects all documents.  
  - `[ _type == "post" ]` – filter clause (like a WHERE).  
  - `{ title, slug, "authorName": author->name }` – projection block; `->` deref

In Strands Agents, callback_handler=None tells the agent not to use a callback handler for processing events during execution. In your case, it prevents the default handler from printing streamed response text automatically, so print(response) doesn't display the same answer a second time


In [2]:
from datetime import datetime
from zoneinfo import ZoneInfo
from strands_tools import calculator

from strands import tool


@tool
def current_time(timezone: str = "Asia/Kolkata") -> str:
    """Get the current date and time in a given IANA timezone.

    Args:
        timezone: IANA timezone name, for example "Asia/Kolkata" or "America/New_York".
    """
    return datetime.now(ZoneInfo(timezone)).strftime("%A, %d %B %Y %I:%M %p %Z")


agent = Agent(model=model, tools=[calculator, current_time], callback_handler= None)

result = agent(
    "What time is it right now in Bangalore?"
    )

print(result.message["content"][0]["text"])



It’s currently **Friday, 30 October 2026, 8:05 PM IST** in Bangalore.


In [ ]:

import requests
from strands import tool


@tool
def get_weather(city: str) -> str:
    """Get today's weather forecast and umbrella advice for a city.

    Args:
        city: City name, for example Pune, Bangalore, or Chennai.
    """
    try:
        # Step 1: Find the city's coordinates
        geo_response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={
                "name": city,
                "count": 1,
                "language": "en",
                "format": "json",
            },
            timeout=10,
        )
        geo_response.raise_for_status()
        locations = geo_response.json().get("results", [])

        if not locations:
            return f"Could not find the city: {city}"

        location = locations[0]
        latitude = location["latitude"]
        longitude = location["longitude"]
        city_name = location["name"]
        country = location.get("country", "")

        # Step 2: Get today's local weather forecast
        weather_response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "daily": [
                    "weather_code",
                    "precipitation_probability_max",
                    "precipitation_sum",
                ],
                "current": [
                    "temperature_2m",
                    "precipitation",
                    "weather_code",
                ],
                "timezone": "auto",
                "forecast_days": 1,
            },
            timeout=10,
        )
        weather_response.raise_for_status()
        data = weather_response.json()

        current = data["current"]
        daily = data["daily"]

        rain_probability = daily["precipitation_probability_max"][0]
        rainfall = daily["precipitation_sum"][0]
        temperature = current["temperature_2m"]
        forecast_date = daily["time"][0]

        # Step 3: Generate a simple recommendation
        if rain_probability is None or rainfall is None:
            advice = "Rain forecast data is unavailable."
        elif rain_probability >= 40 or rainfall >= 0.5:
            advice = "Yes, carrying an umbrella is recommended."
        else:
            advice = "An umbrella is probably not necessary."

        return (
            f"Weather for {city_name}, {country}\n"
            f"Date: {forecast_date}\n"
            f"Current temperature: {temperature}°C\n"
            f"Rain probability today: {rain_probability}%\n"
            f"Expected rainfall today: {rainfall} mm\n"
            f"Recommendation: {advice}"
        )

    except requests.RequestException as e:
        return f"Weather API request failed: {e}"
    except (KeyError, IndexError, TypeError, ValueError) as e:
        return f"Could not process the weather data: {e}"






agent = Agent(
    model=model,
    tools=[get_weather],
    system_prompt=(
        "You are a helpful weather assistant. "
        "Use the get_weather tool to answer weather questions. "
        "Base umbrella recommendations on the returned forecast."
    ),
)

response = agent("Do I need an umbrella in Pune today?")
print(response)



Tool #1: get_weather
Yes—today’s forecast for Pune shows a 55 % chance of rain with about 0.7 mm expected. It’s a good idea to carry an umbrella.Yes—today’s forecast for Pune shows a 55 % chance of rain with about 0.7 mm expected. It’s a good idea to carry an umbrella.

